# 01. 판다스(pandas) 기초 — 데이터를 읽고, 보고, 고르고, 합치기

> **배우는 것**: `read_csv` · `read_json` · `shape` · `info` · `isnull` · `value_counts` · `loc/iloc` · 조건 필터 · `groupby` · `sort_values` · `drop` · `astype` · `apply/map` · `merge` · `concat` · `to_csv`
> ⏱ 예상 소요시간: 60분  |  📑 같이 볼 슬라이드: `slides/01_파이썬_판다스_기초.html`
> 📂 데이터: `telco_churn.csv` (통신사 고객 이탈), `customer_info.json` + `customer_usage.csv` (병합 연습)

**DataFrame 은 엑셀 시트**라고 생각하면 쉬워요. 행(row) = 고객 한 명, 열(column) = 고객의 특징(가입기간, 요금 …).
pandas 는 이 시트를 코드로 다루는 도구입니다.

In [ ]:
# ▶ 맨 처음 한 번 실행하세요 (데이터 경로 + 자동 채점 도구 준비)
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '../data/'          # 데이터 폴더 위치
import aice_check as check     # check.p01_q1() 처럼 쓰는 채점 도구

## 1. 불러오기 (시험 1~3번 유형)

### ✏️ 문제 1. import
pandas 는 **`pd`**, numpy 는 **`np`** 로 불러오세요.

In [ ]:
import pandas as ___
import numpy as ___

<details><summary>💡 힌트 (막히면 클릭)</summary>

`import pandas as pd`

</details>

In [ ]:
check.p01_q1()

### ✏️ 문제 2. CSV 파일 읽기
`DATA_DIR + 'telco_churn.csv'` 를 읽어서 **`df`** 에 저장하세요.

> 시험에서는 `df = pd.read_csv('파일명.csv')` 처럼 파일명만 쓰면 돼요. JSON 이면 `pd.read_json`.

In [ ]:
df = pd.___(DATA_DIR + 'telco_churn.csv')
df.head()      # 위에서 5줄 미리보기 (tail() 은 아래에서 5줄)

<details><summary>💡 힌트 (막히면 클릭)</summary>

`pd.read_csv(경로)`

</details>

In [ ]:
check.p01_q2()

## 2. 데이터 훑어보기 — EDA 의 첫걸음
데이터를 받으면 **항상 이 순서**로 봐요: `shape` → `info()` → `isnull().sum()` → `describe()` → `value_counts()`

### ✏️ 문제 3. 크기 shape
`df.shape` 는 **(행 수, 열 수)** 튜플이에요. 행 수를 **`n_rows`**, 열 수를 **`n_cols`** 에 저장하세요.

In [ ]:
n_rows, n_cols = df.___
print(n_rows, n_cols)

<details><summary>💡 힌트 (막히면 클릭)</summary>

`df.shape` 는 괄호 없는 '속성'이에요. 튜플은 `a, b = (1, 2)` 처럼 나눠 담을 수 있어요.

</details>

In [ ]:
check.p01_q3()

`info()` 는 **컬럼별 타입(Dtype)과 비어 있지 않은 개수(Non-Null)** 를 보여줘요. 아래 셀을 실행하고 두 가지를 찾아보세요.
- `TotalCharges` 는 요금(숫자)인데 Dtype 이 **object(문자열)** 예요. → 숫자로 바꿔야 함 (문제 15)
- `DeviceProtection` 의 Non-Null 개수가 유난히 적어요. → 결측치가 많음

In [ ]:
df.info()

### ✏️ 문제 4. 컬럼별 결측치 개수
컬럼별 결측치(NaN) 개수를 **`null_counts`** 에 저장하세요.

In [ ]:
null_counts = df.___.___
null_counts

<details><summary>💡 힌트 (막히면 클릭)</summary>

`isnull()` 은 칸마다 True/False 를 만들고, `sum()` 은 True 를 1 로 세요. (`isna()` 도 같아요)

</details>

In [ ]:
check.p01_q4()

In [ ]:
df.describe()     # 숫자 컬럼의 평균/표준편차/최소/사분위/최대 — max 가 이상하게 크면 이상치 의심!

### ✏️ 문제 5. 값 분포 value_counts
타깃 컬럼 `Churn` (이탈 여부) 의 값별 개수를 **`churn_counts`** 에 저장하세요.

In [ ]:
churn_counts = df['Churn'].___
churn_counts

<details><summary>💡 힌트 (막히면 클릭)</summary>

`df['컬럼'].value_counts()`  — 비율로 보려면 `value_counts(normalize=True)`

</details>

In [ ]:
check.p01_q5()

### ✏️ 문제 6. 고유값 개수 nunique
`PaymentMethod` 컬럼에 **서로 다른 값이 몇 종류**인지 **`n_payment`** 에 저장하세요.

In [ ]:
print(df['PaymentMethod'].unique())       # 고유값 목록
n_payment = df['PaymentMethod'].___  # 고유값 개수
n_payment

<details><summary>💡 힌트 (막히면 클릭)</summary>

unique = 목록, nunique = 개수 (number of unique)

</details>

In [ ]:
check.p01_q6()

## 3. 원하는 부분만 고르기
| 방법 | 기준 | 예 | 끝 포함? |
|---|---|---|---|
| `df['a']` | 컬럼 1개 → Series | `df['tenure']` | |
| `df[['a','b']]` | 컬럼 여러 개 → DataFrame | 대괄호 **두 겹** | |
| `df.loc[행, 열]` | **이름(라벨)** | `df.loc[0:2, 'tenure']` | **포함** |
| `df.iloc[행, 열]` | **위치(번호)** | `df.iloc[0:5, 0:3]` | 미포함 |
| `df[조건]` | True 인 행만 | `df[df['tenure'] >= 60]` | |

### ✏️ 문제 7. 여러 컬럼 선택
`customerID`, `tenure`, `Churn` 세 컬럼만 골라 **`sub`** 에 저장하세요.

In [ ]:
sub = df___
sub.head(3)

<details><summary>💡 힌트 (막히면 클릭)</summary>

리스트를 대괄호 안에 넣으니 `[[ ... ]]` 두 겹이 돼요.

</details>

In [ ]:
check.p01_q7()

### ✏️ 문제 8. loc
`loc` 로 0~2번 행의 `tenure` 값을 골라 **`first3_tenure`** 에 저장하세요.

In [ ]:
first3_tenure = df.___[0:2, 'tenure']
first3_tenure

<details><summary>💡 힌트 (막히면 클릭)</summary>

loc 는 라벨 기준이라 0:2 가 0, 1, 2 **세 개**예요.

</details>

In [ ]:
check.p01_q8()

### ✏️ 문제 9. iloc
`iloc` 로 앞 5행 × 앞 3열을 골라 **`block`** 에 저장하세요.

In [ ]:
block = df.___[0:5, 0:3]
block

<details><summary>💡 힌트 (막히면 클릭)</summary>

iloc 는 파이썬 슬라이싱처럼 끝 번호를 포함하지 않아요.

</details>

In [ ]:
check.p01_q9()

### ✏️ 문제 10. 조건 필터링
가입기간 `tenure` 가 **60개월 이상**인 고객만 **`long_customers`** 에 저장하세요.

In [ ]:
long_customers = df[df['tenure'] ___]
len(long_customers)

<details><summary>💡 힌트 (막히면 클릭)</summary>

`df[df['컬럼'] >= 값]` — 안쪽 `df['tenure'] >= 60` 이 True/False 목록을 만들어요.

</details>

In [ ]:
check.p01_q10()

### ✏️ 문제 11. 여러 조건 (그리고 &)
**월 단위 계약**(`Contract == 'Month-to-month'`) **이면서** 가입 **12개월 미만**인 고객을 **`risky`** 에 저장하세요.

In [ ]:
risky = df[(df['Contract'] == 'Month-to-month') ___ (df['tenure'] < 12)]
len(risky)

<details><summary>💡 힌트 (막히면 클릭)</summary>

pandas 조건 연결: 그리고 `&`, 또는 `|`, 아니다 `~` — **조건마다 괄호** 필수! (`and` 쓰면 에러)

</details>

In [ ]:
check.p01_q11()

## 4. 묶어서 계산하고 정렬하기

### ✏️ 문제 12. groupby
계약 형태(`Contract`)별 월 요금(`MonthlyCharges`) **평균**을 **`mean_charge`** 에 저장하세요.

> 엑셀의 **피벗 테이블**과 같아요: `df.groupby('묶을 컬럼')['계산할 컬럼'].집계함수()`

In [ ]:
mean_charge = df.___['MonthlyCharges'].___
mean_charge

<details><summary>💡 힌트 (막히면 클릭)</summary>

집계함수: mean, sum, count, max, min, median

</details>

In [ ]:
check.p01_q12()

### ✏️ 문제 13. 정렬 sort_values
`tenure` 가 **큰 순서**로 정렬해서 위 10명을 **`top_tenure`** 에 저장하세요.

In [ ]:
top_tenure = df.sort_values('tenure', ascending=___).head(10)
top_tenure[['customerID', 'tenure']]

<details><summary>💡 힌트 (막히면 클릭)</summary>

ascending=True 가 기본(작은 순). 큰 순서는 False.

</details>

In [ ]:
check.p01_q13()

## 5. 바꾸기 — 컬럼 삭제, 타입 변환, 값 변환
> ⚠️ **중요한 습관**: pandas 함수 대부분은 원본을 바꾸지 않고 **새 결과를 돌려줘요.** 결과를 꼭 변수에 다시 담으세요.
> `df.drop('a', axis=1)` 만 쓰면 아무 일도 안 일어난 것처럼 보여요 → `df = df.drop('a', axis=1)`
> (`inplace=True` 옵션도 있지만, 최신 pandas 에서는 **다시 대입하는 방식**을 권장해요.)

### ✏️ 문제 14. 컬럼 삭제 drop
`customerID` 컬럼을 삭제한 결과를 **`df2`** 에 저장하세요. (원본 `df` 는 그대로 두기)

In [ ]:
df2 = df.drop('customerID', axis=___)
df2.columns

<details><summary>💡 힌트 (막히면 클릭)</summary>

axis=0 은 행(위아래), axis=1 은 열(좌우). 컬럼 삭제는 axis=1

</details>

In [ ]:
check.p01_q14()

### `TotalCharges` 가 문자열인 이유 찾기
숫자로 바꾸려고 `astype(float)` 를 하면 **에러**가 나요. 아래 셀을 실행해 이유를 확인해 보세요.

In [ ]:
try:
    df['TotalCharges'].astype(float)
except Exception as e:
    print('에러:', e)

# 범인은 ' ' (공백 한 칸)! 가입한 지 0개월인 고객은 누적 요금이 공백으로 들어 있어요.
df[df['TotalCharges'] == ' '][['tenure', 'MonthlyCharges', 'TotalCharges']].head()

### ✏️ 문제 15. 문자열 → 숫자
`' '` 를 `'0'` 으로 바꾼 뒤 `float` 로 변환해서 **`df['TotalCharges']`** 에 다시 저장하세요.

In [ ]:
df['TotalCharges'] = df['TotalCharges'].___.___
df['TotalCharges'].dtype

<details><summary>💡 힌트 (막히면 클릭)</summary>

`.replace(바꿀값, 새값)` 다음에 `.astype(float)` 를 이어서(체이닝) 쓰세요.

다른 방법: `pd.to_numeric(df['TotalCharges'], errors='coerce')` → 숫자로 못 바꾸는 값은 NaN (03 전처리에서 배워요)

</details>

In [ ]:
check.p01_q15()

### ✏️ 문제 16. apply + lambda
`tenure`(개월)를 **년 단위**(12로 나눈 몫)로 바꾼 새 컬럼 **`df['tenure_year']`** 를 만드세요.

> `lambda x: x // 12` 는 "x 를 받아서 x//12 를 돌려주는 이름 없는 함수"예요. `apply` 가 값 하나하나에 적용해 줍니다.

In [ ]:
df['tenure_year'] = df['tenure'].apply(___)
df[['tenure', 'tenure_year']].head()

<details><summary>💡 힌트 (막히면 클릭)</summary>

`//` 는 몫, `%` 는 나머지

</details>

In [ ]:
check.p01_q16()

### ✏️ 문제 17. map 으로 값 바꾸기
`Churn` 의 `'Yes'` → `1`, `'No'` → `0` 으로 바꾼 새 컬럼 **`df['Churn_num']`** 을 만드세요.

In [ ]:
df['Churn_num'] = df['Churn'].map(___)
df['Churn_num'].value_counts()

<details><summary>💡 힌트 (막히면 클릭)</summary>

map 에 딕셔너리 {원래값: 새값} 을 넘겨요. 딕셔너리에 없는 값은 NaN 이 되니 주의!

</details>

In [ ]:
check.p01_q17()

## 6. 합치기 — merge / concat (실제 시험에 자주 나와요!)
- **`pd.merge(왼쪽, 오른쪽, on='키', how='inner')`** : 같은 키(고객ID)를 기준으로 **옆으로** 붙이기 (SQL JOIN)
- **`pd.concat([df1, df2])`** : 같은 모양의 표를 **위아래로** 쌓기

| how | 남는 행 |
|---|---|
| `inner` (기본값) | 양쪽에 **모두** 있는 키만 |
| `left` | 왼쪽 표의 키는 **전부** (오른쪽에 없으면 NaN) |
| `right` | 오른쪽 표의 키는 전부 |
| `outer` | 어느 한쪽에라도 있으면 전부 |

### ✏️ 문제 18. JSON 과 CSV 읽기
`customer_info.json` 을 **`info`** 에, `customer_usage.csv` 를 **`usage`** 에 읽어오세요.

In [ ]:
info = pd.___(DATA_DIR + 'customer_info.json')
usage = pd.___(DATA_DIR + 'customer_usage.csv')
print(info.shape, usage.shape)
info.head(3)

<details><summary>💡 힌트 (막히면 클릭)</summary>

JSON 파일은 `pd.read_json`

</details>

In [ ]:
check.p01_q18()

### ✏️ 문제 19. merge (inner)
`customerID` 기준으로 **inner** 병합해 **`merged`** 에 저장하세요.

In [ ]:
merged = pd.merge(info, usage, on=___, how=___)
merged.shape

<details><summary>💡 힌트 (막히면 클릭)</summary>

`on='공통 컬럼명'`, `how='inner'`

</details>

In [ ]:
check.p01_q19()

### ✏️ 문제 20. merge (left)
같은 조건으로 **left** 병합해 **`merged_left`** 에 저장하고, 몇 행이 NaN 이 생겼는지 보세요.

In [ ]:
merged_left = pd.merge(info, usage, on='customerID', how=___)
print(merged_left.shape)
merged_left.isnull().sum()

<details><summary>💡 힌트 (막히면 클릭)</summary>

how 만 바꾸면 돼요.

</details>

In [ ]:
check.p01_q20()

### ✏️ 문제 21. concat
`df.head(100)` 과 `df.tail(100)` 을 위아래로 붙여 **`stacked`** 에 저장하세요. 인덱스는 0부터 새로 매기세요.

In [ ]:
stacked = pd.___([df.head(100), df.tail(100)], ignore_index=___)
stacked.shape

<details><summary>💡 힌트 (막히면 클릭)</summary>

`pd.concat([표1, 표2], ignore_index=True)` — 리스트로 넘기는 것 주의!

</details>

In [ ]:
check.p01_q21()

## 7. 저장하기

### ✏️ 문제 22. CSV 로 저장
`merged` 를 **`merged_result.csv`** 파일로 저장하세요. 인덱스는 저장하지 않습니다.

In [ ]:
merged.to_csv('merged_result.csv', index=___)
pd.read_csv('merged_result.csv').head(3)

<details><summary>💡 힌트 (막히면 클릭)</summary>

index=False 를 빼면 다시 읽을 때 `Unnamed: 0` 이라는 쓸모없는 컬럼이 생겨요.

</details>

In [ ]:
check.p01_q22()

## 🏁 마무리

In [ ]:
check.progress()

### 오늘 배운 것 한눈에
| 하고 싶은 것 | 코드 |
|---|---|
| 읽기 | `pd.read_csv()`, `pd.read_json()` |
| 훑어보기 | `shape`, `info()`, `describe()`, `head()` |
| 결측 / 분포 | `isnull().sum()`, `value_counts()`, `nunique()` |
| 고르기 | `df[['a','b']]`, `loc`, `iloc`, `df[(조건1) & (조건2)]` |
| 묶기 / 정렬 | `groupby('a')['b'].mean()`, `sort_values('a', ascending=False)` |
| 바꾸기 | `drop(axis=1)`, `replace`, `astype`, `apply(lambda)`, `map({})` |
| 합치기 | `pd.merge(on=, how=)`, `pd.concat([..], ignore_index=True)` |
| 저장 | `to_csv('파일', index=False)` |

다음: **`02_EDA_시각화_실습.ipynb`**